# Trabajo 3: Análisis de datos con NumPy y Pandas

## RetailNow — Análisis de ventas, inventarios y satisfacción del cliente

Este notebook procesa los archivos `sales.csv`, `inventories.csv` y `satisfaction.csv` con **Pandas** y realiza cálculos estadísticos y simulaciones con **NumPy**.

Los análisis incluidos cumplen con los puntos solicitados en el ejercicio:

- carga y limpieza de los datos;
- ventas totales por producto y por tienda;
- ingresos totales por tienda;
- resumen estadístico;
- rotación de inventarios e inventarios críticos;
- satisfacción del cliente y relación con las ventas;
- mediana y desviación estándar con NumPy;
- simulación reproducible de ventas futuras.

## 1. Importar las librerías necesarias

In [ ]:
from pathlib import Path

import numpy as np
import pandas as pd

## 2. Cargar los archivos CSV

La plataforma de evaluación indica las rutas absolutas `/workspace/...`.
Para poder ejecutar el mismo notebook localmente en Visual Studio Code, si esas rutas no existen se utilizará la carpeta `data/` ubicada junto al notebook.

In [ ]:
# Rutas requeridas por la plataforma.
ruta_sales = Path("/workspace/sales.csv")
ruta_inventories = Path("/workspace/inventories.csv")
ruta_satisfaction = Path("/workspace/satisfaction.csv")

# Rutas locales del proyecto para trabajar en Visual Studio Code.
if not ruta_sales.exists():
    ruta_sales = Path("data/sales.csv")
    ruta_inventories = Path("data/inventories.csv")
    ruta_satisfaction = Path("data/satisfaction.csv")

ventas = pd.read_csv(ruta_sales)
inventarios = pd.read_csv(ruta_inventories)
satisfaccion = pd.read_csv(ruta_satisfaction)

print("DataFrame de ventas:")
display(ventas.head())

print("\nDataFrame de inventarios:")
display(inventarios.head())

print("\nDataFrame de satisfacción:")
display(satisfaccion.head())

## 3. Verificar la estructura y limpiar los datos

Los archivos reales proporcionados utilizan estas columnas:

- `sales.csv`: `ID_Tienda`, `Producto`, `Cantidad_Vendida`, `Precio_Unitario`, `Fecha_Venta`
- `inventories.csv`: `ID_Tienda`, `Producto`, `Stock_Disponible`, `Fecha_Actualización`
- `satisfaction.csv`: `ID_Tienda`, `Satisfacción_Promedio`, `Fecha_Evaluación`

Se convierten las fechas a `datetime` y se eliminan las filas con valores nulos mediante `dropna()`.

In [ ]:
print("Columnas de ventas:", ventas.columns.tolist())
print("Columnas de inventarios:", inventarios.columns.tolist())
print("Columnas de satisfacción:", satisfaccion.columns.tolist())

print("\nValores nulos antes de limpiar:")
print("Ventas:", ventas.isna().sum().sum())
print("Inventarios:", inventarios.isna().sum().sum())
print("Satisfacción:", satisfaccion.isna().sum().sum())

In [ ]:
# Convertir las columnas de fecha al tipo datetime.
ventas["Fecha_Venta"] = pd.to_datetime(ventas["Fecha_Venta"])
inventarios["Fecha_Actualización"] = pd.to_datetime(
    inventarios["Fecha_Actualización"]
)
satisfaccion["Fecha_Evaluación"] = pd.to_datetime(
    satisfaccion["Fecha_Evaluación"]
)

# Eliminar filas con valores nulos.
ventas = ventas.dropna().copy()
inventarios = inventarios.dropna().copy()
satisfaccion = satisfaccion.dropna().copy()

print("Filas válidas después de aplicar dropna():")
print("Ventas:", len(ventas))
print("Inventarios:", len(inventarios))
print("Satisfacción:", len(satisfaccion))

## 4. Exploración y análisis de ventas con Pandas

Primero se crea la columna `Total_Ventas`, calculada como:

**Cantidad_Vendida × Precio_Unitario**

Después se calculan las ventas totales por producto, las ventas totales por tienda y los ingresos totales por tienda.

In [ ]:
# Crear la columna monetaria solicitada para el análisis.
ventas["Total_Ventas"] = (
    ventas["Cantidad_Vendida"] * ventas["Precio_Unitario"]
)

print("Ventas con la columna Total_Ventas:")
display(ventas)

In [ ]:
# Ventas totales por producto, medidas en unidades vendidas.
ventas_por_producto = (
    ventas.groupby("Producto", as_index=False)["Cantidad_Vendida"]
    .sum()
    .rename(columns={"Cantidad_Vendida": "Unidades_Totales_Vendidas"})
    .sort_values("Unidades_Totales_Vendidas", ascending=False)
)

print("Ventas totales por producto:")
display(ventas_por_producto)

In [ ]:
# Ventas totales por tienda, medidas en unidades vendidas.
ventas_por_tienda = (
    ventas.groupby("ID_Tienda", as_index=False)["Cantidad_Vendida"]
    .sum()
    .rename(columns={"Cantidad_Vendida": "Unidades_Totales_Vendidas"})
    .sort_values("Unidades_Totales_Vendidas", ascending=False)
)

print("Ventas totales por tienda:")
display(ventas_por_tienda)

In [ ]:
# Ingresos totales por tienda.
ingresos_por_tienda = (
    ventas.groupby("ID_Tienda", as_index=False)["Total_Ventas"]
    .sum()
    .rename(columns={"Total_Ventas": "Ingresos_Totales"})
    .sort_values("Ingresos_Totales", ascending=False)
)

print("Ingresos totales por tienda:")
display(ingresos_por_tienda)

### Resumen estadístico de las ventas

`describe()` permite obtener métricas como cantidad, media, desviación estándar, mínimo, percentiles y máximo.

El archivo `sales.csv` proporcionado **no contiene una columna de categoría de producto**, por lo que el análisis opcional de promedio por tienda y categoría no puede realizarse con estos datos sin inventar información.

In [ ]:
print("Resumen estadístico:")
display(
    ventas[
        ["Cantidad_Vendida", "Precio_Unitario", "Total_Ventas"]
    ].describe()
)

print("Mediana de Total_Ventas con Pandas:", ventas["Total_Ventas"].median())

## 5. Análisis de inventarios con Pandas

La rotación de inventario se calcula, para cada combinación de tienda y producto, como:

**Cantidad vendida / Stock disponible**

Según el enunciado, se considera crítico un registro cuya proporción vendida sea inferior al **10 %** del stock disponible.

In [ ]:
# Sumar las unidades vendidas por tienda y producto.
ventas_tienda_producto = (
    ventas.groupby(
        ["ID_Tienda", "Producto"],
        as_index=False
    )["Cantidad_Vendida"].sum()
)

# Relacionar ventas e inventarios.
inventarios_analisis = inventarios.merge(
    ventas_tienda_producto,
    on=["ID_Tienda", "Producto"],
    how="left"
)

# Si un producto no tiene ventas registradas, se consideran 0 unidades.
inventarios_analisis["Cantidad_Vendida"] = (
    inventarios_analisis["Cantidad_Vendida"].fillna(0)
)

# Calcular la rotación evitando divisiones entre cero.
inventarios_analisis["Rotacion_Inventario"] = np.where(
    inventarios_analisis["Stock_Disponible"] > 0,
    inventarios_analisis["Cantidad_Vendida"]
    / inventarios_analisis["Stock_Disponible"],
    np.nan
)

# Mostrar también la rotación como porcentaje para facilitar la lectura.
inventarios_analisis["Rotacion_Porcentaje"] = (
    inventarios_analisis["Rotacion_Inventario"] * 100
)

print("Rotación de inventario por tienda y producto:")
display(inventarios_analisis)

In [ ]:
# Filtrar registros con rotación inferior al 10 %.
inventarios_criticos = inventarios_analisis[
    inventarios_analisis["Rotacion_Inventario"] < 0.10
].copy()

print("Inventarios críticos (< 10 %):")

if inventarios_criticos.empty:
    print("No se encontraron productos con rotación inferior al 10 %.")
else:
    display(inventarios_criticos)

In [ ]:
# Calcular la rotación promedio por tienda utilizando groupby().
rotacion_por_tienda = (
    inventarios_analisis.groupby(
        "ID_Tienda",
        as_index=False
    )["Rotacion_Inventario"]
    .mean()
)

rotacion_por_tienda["Rotacion_Porcentaje"] = (
    rotacion_por_tienda["Rotacion_Inventario"] * 100
)

print("Rotación promedio de inventario por tienda:")
display(rotacion_por_tienda)

## 6. Satisfacción del cliente

Se analiza la satisfacción promedio de cada tienda y se relaciona con los ingresos totales obtenidos.

El archivo entregado expresa la satisfacción en una escala de **0 a 100**, por lo que se filtran las tiendas con una satisfacción inferior a **60**.

In [ ]:
# El archivo ya contiene una satisfacción promedio por tienda.
satisfaccion_por_tienda = satisfaccion[
    ["ID_Tienda", "Satisfacción_Promedio"]
].copy()

print("Satisfacción por tienda:")
display(satisfaccion_por_tienda)

In [ ]:
# Relacionar satisfacción e ingresos.
rendimiento_tiendas = ingresos_por_tienda.merge(
    satisfaccion_por_tienda,
    on="ID_Tienda",
    how="inner"
)

print("Relación entre ingresos y satisfacción:")
display(rendimiento_tiendas)

In [ ]:
# Filtrar tiendas con satisfacción menor al 60 %.
tiendas_baja_satisfaccion = rendimiento_tiendas[
    rendimiento_tiendas["Satisfacción_Promedio"] < 60
].copy()

print("Tiendas con satisfacción menor al 60 %:")
display(tiendas_baja_satisfaccion)

### Recomendaciones

Para las tiendas con satisfacción inferior al 60 %, se recomienda revisar la atención al cliente, la disponibilidad de productos y los tiempos de servicio. También conviene analizar comentarios de clientes y comparar su rendimiento comercial con el resto de las sucursales.

In [ ]:
if tiendas_baja_satisfaccion.empty:
    print("No se encontraron tiendas con satisfacción inferior al 60 %.")
else:
    for _, fila in tiendas_baja_satisfaccion.iterrows():
        print(
            f"Tienda {int(fila['ID_Tienda'])}: "
            f"satisfacción = {fila['Satisfacción_Promedio']} %, "
            f"ingresos = ${fila['Ingresos_Totales']:,.2f}. "
            "Recomendación: mejorar atención al cliente, revisar "
            "disponibilidad de productos y tiempos de servicio."
        )

## 7. Operaciones con NumPy

Para cumplir expresamente con el requisito del ejercicio, la columna `Total_Ventas` se convierte a un array de NumPy mediante `.to_numpy()`.

Sobre este array se calculan:

- la mediana;
- la desviación estándar.

In [ ]:
# Convertir Total_Ventas de Pandas a un array NumPy.
ventas_numpy = ventas["Total_Ventas"].to_numpy()

mediana_ventas_numpy = np.median(ventas_numpy)
desviacion_estandar_numpy = np.std(ventas_numpy)

print("Array NumPy de Total_Ventas:")
print(ventas_numpy)

print("\nMediana de las ventas totales:", mediana_ventas_numpy)
print("Desviación estándar de las ventas:", desviacion_estandar_numpy)

## 8. Simulación de proyecciones de ventas futuras con NumPy

Se establece una semilla para obtener resultados reproducibles.

Como simulación sencilla, cada venta actual se multiplica por un factor aleatorio con media `1.05` (crecimiento esperado aproximado del 5 %) y desviación estándar `0.10`.

In [ ]:
# Generador aleatorio reproducible.
rng = np.random.default_rng(seed=42)

# Factores aleatorios para simular ventas futuras.
factores_proyeccion = rng.normal(
    loc=1.05,
    scale=0.10,
    size=ventas_numpy.size
)

ventas_proyectadas = ventas_numpy * factores_proyeccion

print("Ventas actuales:")
print(ventas_numpy)

print("\nVentas futuras simuladas:")
print(np.round(ventas_proyectadas, 2))

In [ ]:
print("Estadísticas de las ventas futuras simuladas:")
print("Media:", round(np.mean(ventas_proyectadas), 2))
print("Mediana:", round(np.median(ventas_proyectadas), 2))
print(
    "Desviación estándar:",
    round(np.std(ventas_proyectadas), 2)
)
print("Máximo:", round(np.max(ventas_proyectadas), 2))
print("Mínimo:", round(np.min(ventas_proyectadas), 2))

## 9. Conclusiones

El análisis permitió integrar los datos reales de ventas, inventarios y satisfacción de RetailNow.

- Se calcularon las unidades vendidas por producto y por tienda.
- Se calcularon los ingresos totales mediante `Cantidad_Vendida × Precio_Unitario`.
- Se obtuvo un resumen estadístico de las ventas.
- Se calculó la rotación de inventario por tienda y producto y se aplicó el filtro de nivel crítico inferior al 10 %.
- Se relacionó la satisfacción de cada tienda con sus ingresos y se identificaron las tiendas con satisfacción inferior al 60 %.
- Con NumPy se calculó la mediana y la desviación estándar de `Total_Ventas`.
- Se realizó una simulación reproducible de ventas futuras usando un generador aleatorio con semilla fija.

El archivo de ventas proporcionado no contiene una columna de categoría, por lo que no se realizó el análisis opcional por categoría para evitar inventar información que no existe en los datos.